# 1.1 — Install PySpark-Samar Wazir
Install Java (required by Spark) and PySpark, then configure the environment.

In [ ]:
# Install Java first (fixes JAVA_GATEWAY_EXITED error)
!apt-get install openjdk-11-jdk -q
!pip install pyspark==3.5.1 -q

import os, pyspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = "python3"
print(f"PySpark {pyspark.__version__} ready")

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk-headless openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libxt-doc openjdk-11-demo openjdk-11-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-11-jdk openjdk-11-jdk-headless openjdk-11-jre
  openjd

# 1.2 — Create SparkSession
`SparkSession` is the entry point to all Spark functionality. `local[*]` uses all available CPU cores.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("LNDN5204-Week4") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | App: {spark.sparkContext.appName}")

Spark 3.5.1 | App: LNDN5204-Week4


# 2.1 — Dataset: Supermarket Sales
**Source:** [Kaggle — Supermarket Sales](https://www.kaggle.com/datasets/aungpyaeap/supermarket-sales)  
**Size:** ~1,000 rows × 17 columns | 3 branches, 3 cities, 3 months of transactions.  
**Key columns:** Branch, City, Customer_type, Product_line, Quantity, Total, Rating, Payment

# 2.2 — Kaggle API Setup
Get your API key: kaggle.com → Profile → Settings → API → Create New Token → upload `kaggle.json` below.

In [ ]:
!pip install kaggle -q

from google.colab import files
import os, shutil

uploaded = files.upload()  # upload kaggle.json when prompted
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle API key configured")

Saving kaggle.json to kaggle.json
Kaggle API key configured


# 2.3 — Download & Extract Dataset

In [ ]:
import zipfile, shutil

!kaggle datasets download -d faresashraf1001/supermarket-sales --force -q

with zipfile.ZipFile("supermarket-sales.zip", "r") as z:
    z.extractall("supermarket_data")

shutil.copy("supermarket_data/SuperMarket Analysis.csv", "supermarket_sales.csv")
print("Dataset ready: supermarket_sales.csv")

Dataset URL: https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales
License(s): apache-2.0
Dataset ready: supermarket_sales.csv


# 3.1 — Load CSV into Spark DataFrame
`inferSchema=True` auto-detects column types. `header=True` uses the first row as column names.

In [ ]:
df = spark.read.csv("supermarket_sales.csv", header=True, inferSchema=True)
print(f"Rows: {df.count():,} | Columns: {len(df.columns)}")
print(df.columns)

Rows: 1,000 | Columns: 17
['Invoice ID', 'Branch', 'City', 'Customer type', 'Gender', 'Product line', 'Unit price', 'Quantity', 'Tax 5%', 'Sales', 'Date', 'Time', 'Payment', 'cogs', 'gross margin percentage', 'gross income', 'Rating']


# 3.2 — Clean Column Names
Replace spaces with underscores so column names work in Spark SQL and DataFrame operations.

In [ ]:
clean_cols = [c.strip().replace(" ", "_").replace("%", "pct") for c in df.columns]
df = df.toDF(*clean_cols)
print("Cleaned columns:", df.columns)

Cleaned columns: ['Invoice_ID', 'Branch', 'City', 'Customer_type', 'Gender', 'Product_line', 'Unit_price', 'Quantity', 'Tax_5pct', 'Sales', 'Date', 'Time', 'Payment', 'cogs', 'gross_margin_percentage', 'gross_income', 'Rating']


# 4.1 — Inspect Schema
Always check the schema after loading. Watch for dates loaded as String and numbers as String.

In [ ]:
df.printSchema()

root
 |-- Invoice_ID: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_type: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Product_line: string (nullable = true)
 |-- Unit_price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Tax_5pct: double (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Payment: string (nullable = true)
 |-- cogs: double (nullable = true)
 |-- gross_margin_percentage: double (nullable = true)
 |-- gross_income: double (nullable = true)
 |-- Rating: double (nullable = true)



# 4.3 — Fix Date Column
`Date` was loaded as String (e.g. `1/5/2019`). Convert it to `DateType` for time-based analysis.

In [ ]:
from pyspark.sql.functions import to_date, col

df = df.withColumn("Date", to_date(col("Date"), "M/d/yyyy"))
df.select("Date").show(3)
print(f"Date type: {dict(df.dtypes)['Date']}")

+----------+
|      Date|
+----------+
|2019-01-05|
|2019-03-08|
|2019-03-03|
+----------+
only showing top 3 rows

Date type: date


# 5.1 — Preview the DataFrame
`.show()` prints rows to screen. `.first()` returns a single Row object.

In [ ]:
print("First 5 rows:")
df.select("Invoice_ID","Branch","City","Product_line","Sales","Rating").show(5, truncate=True)

First 5 rows:
+-----------+------+---------+--------------------+--------+------+
| Invoice_ID|Branch|     City|        Product_line|   Sales|Rating|
+-----------+------+---------+--------------------+--------+------+
|750-67-8428|  Alex|   Yangon|   Health and beauty|548.9715|   9.1|
|226-31-3081|  Giza|Naypyitaw|Electronic access...|   80.22|   9.6|
|631-41-3108|  Alex|   Yangon|  Home and lifestyle|340.5255|   7.4|
|123-19-1176|  Alex|   Yangon|   Health and beauty| 489.048|   8.4|
|373-73-7910|  Alex|   Yangon|   Sports and travel|634.3785|   5.3|
+-----------+------+---------+--------------------+--------+------+
only showing top 5 rows



In [ ]:
print("Vertical view (first row):")
df.show(1, vertical=True)

Vertical view (first row):
-RECORD 0------------------------------------
 Invoice_ID              | 750-67-8428       
 Branch                  | Alex              
 City                    | Yangon            
 Customer_type           | Member            
 Gender                  | Female            
 Product_line            | Health and beauty 
 Unit_price              | 74.69             
 Quantity                | 7                 
 Tax_5pct                | 26.1415           
 Sales                   | 548.9715          
 Date                    | 2019-01-05        
 Time                    | 1:08:00 PM        
 Payment                 | Ewallet           
 cogs                    | 522.83            
 gross_margin_percentage | 4.761904762       
 gross_income            | 26.1415           
 Rating                  | 9.1               
only showing top 1 row



In [ ]:
first_row = df.first()
print(f"Invoice: {first_row['Invoice_ID']} | Branch: {first_row['Branch']} | Total: {first_row['Sales']}")

Invoice: 750-67-8428 | Branch: Alex | Total: 548.9715


# 6.1 — Summary Statistics
`.describe()` returns count, mean, std, min, max for numeric columns.

In [ ]:
df.select("Unit_price","Quantity","Sales","Rating").describe().show()

+-------+------------------+------------------+------------------+------------------+
|summary|        Unit_price|          Quantity|             Sales|            Rating|
+-------+------------------+------------------+------------------+------------------+
|  count|              1000|              1000|              1000|              1000|
|   mean| 55.67212999999998|              5.51|322.96674900000005| 6.972700000000003|
| stddev|26.494628347919786|2.9234305954556956| 245.8853351009718|1.7185802943791213|
|    min|             10.08|                 1|           10.6785|               4.0|
|    max|             99.96|                10|           1042.65|              10.0|
+-------+------------------+------------------+------------------+------------------+



In [ ]:
from pyspark.sql.functions import countDistinct

for c in ["Branch","City","Customer_type","Gender","Product_line","Payment"]:
    n = df.select(countDistinct(c)).collect()[0][0]
    print(f"  {c:<25} {n} distinct values")

  Branch                    3 distinct values
  City                      3 distinct values
  Customer_type             2 distinct values
  Gender                    2 distinct values
  Product_line              6 distinct values
  Payment                   3 distinct values


In [ ]:
df.groupBy("Branch").count().orderBy("Branch").show()

+------+-----+
|Branch|count|
+------+-----+
|  Alex|  340|
| Cairo|  332|
|  Giza|  328|
+------+-----+



# 7.1 — Selecting Columns
`.select()` chooses columns — equivalent to SQL `SELECT`. Use `col()` for expressions.

In [ ]:
from pyspark.sql.functions import col, round as spark_round

df.select(
    col("Invoice_ID"),
    col("Branch"),
    col("Product_line"),
    spark_round(col("Sales"), 2).alias("Total_USD"),
    col("Rating")
).show(5)

+-----------+------+--------------------+---------+------+
| Invoice_ID|Branch|        Product_line|Total_USD|Rating|
+-----------+------+--------------------+---------+------+
|750-67-8428|  Alex|   Health and beauty|   548.97|   9.1|
|226-31-3081|  Giza|Electronic access...|    80.22|   9.6|
|631-41-3108|  Alex|  Home and lifestyle|   340.53|   7.4|
|123-19-1176|  Alex|   Health and beauty|   489.05|   8.4|
|373-73-7910|  Alex|   Sports and travel|   634.38|   5.3|
+-----------+------+--------------------+---------+------+
only showing top 5 rows



# 7.2 — Filtering Rows
`.filter()` keeps rows matching a condition — equivalent to SQL `WHERE`.

In [ ]:
branch_a = df.filter(df["Branch"] == "Alex")
print(f"Branch Alex transactions: {branch_a.count()}")
branch_a.select("Invoice_ID","City","Product_line","Sales").show(5)

Branch Alex transactions: 340
+-----------+------+--------------------+--------+
| Invoice_ID|  City|        Product_line|   Sales|
+-----------+------+--------------------+--------+
|750-67-8428|Yangon|   Health and beauty|548.9715|
|631-41-3108|Yangon|  Home and lifestyle|340.5255|
|123-19-1176|Yangon|   Health and beauty| 489.048|
|373-73-7910|Yangon|   Sports and travel|634.3785|
|355-53-5943|Yangon|Electronic access...| 433.692|
+-----------+------+--------------------+--------+
only showing top 5 rows



In [ ]:
high_value = df.filter((df["Sales"] > 300) & (df["Branch"].isin("Giza","Cairo")))
print(f"High-value B/C transactions: {high_value.count()}")
high_value.select("Branch","Product_line","Sales","Payment").show(5)

High-value B/C transactions: 290
+------+--------------------+--------+-------+
|Branch|        Product_line|   Sales|Payment|
+------+--------------------+--------+-------+
|  Giza|Electronic access...|627.6165|Ewallet|
|  Giza|  Home and lifestyle|  772.38|Ewallet|
| Cairo|   Sports and travel| 590.436|   Cash|
|  Giza|Electronic access...|  451.71|Ewallet|
| Cairo|  Food and beverages|  463.89|   Cash|
+------+--------------------+--------+-------+
only showing top 5 rows



In [ ]:
premium = df.filter(
    (df["Gender"] == "Female") &
    (df["Customer_type"] == "Member") &
    (df["Payment"] == "Ewallet") &
    (df["Rating"] >= 8)
)
print(f"Premium female members via Ewallet (Rating >= 8): {premium.count()}")
premium.select("Branch","Product_line","Sales","Rating").show(5)

Premium female members via Ewallet (Rating >= 8): 30
+------+--------------------+--------+------+
|Branch|        Product_line|   Sales|Rating|
+------+--------------------+--------+------+
|  Alex|   Health and beauty|548.9715|   9.1|
|  Alex|   Health and beauty| 489.048|   8.4|
|  Giza|  Home and lifestyle|  772.38|   8.0|
|  Alex|  Food and beverages| 453.495|   8.2|
|  Alex|Electronic access...|  181.44|   9.9|
+------+--------------------+--------+------+
only showing top 5 rows



# 7.3 — Sorting Results
`.orderBy()` sorts by one or more columns. Use `.desc()` for descending order.

In [ ]:
df.select("Invoice_ID","Branch","Product_line","Sales","Rating") \
  .orderBy(col("Sales").desc()) \
  .show(10)

+-----------+------+-------------------+--------+------+
| Invoice_ID|Branch|       Product_line|   Sales|Rating|
+-----------+------+-------------------+--------+------+
|860-79-0874|  Giza|Fashion accessories| 1042.65|   6.6|
|687-47-8271|  Alex|Fashion accessories| 1039.29|   8.7|
|283-26-5248|  Giza| Food and beverages| 1034.46|   4.5|
|751-41-9720|  Giza| Home and lifestyle| 1023.75|   8.0|
|303-96-2227| Cairo| Home and lifestyle| 1022.49|   4.4|
|744-16-7898| Cairo| Home and lifestyle|1022.385|   4.9|
|271-88-8734|  Giza|Fashion accessories|1020.705|   8.7|
|234-65-2137|  Giza| Home and lifestyle| 1003.59|   4.8|
|554-42-2417|  Giza|  Sports and travel| 1002.12|   5.2|
|325-77-6186|  Alex| Home and lifestyle| 951.825|   7.3|
+-----------+------+-------------------+--------+------+
only showing top 10 rows



In [ ]:
df.select("Invoice_ID","Branch","Product_line","Sales","Rating") \
  .orderBy(col("Rating").asc()) \
  .show(10)

+-----------+------+--------------------+--------+------+
| Invoice_ID|Branch|        Product_line|   Sales|Rating|
+-----------+------+--------------------+--------+------+
|836-82-5858| Cairo|   Health and beauty|655.5465|   4.0|
|510-95-6347| Cairo|  Food and beverages| 152.838|   4.0|
|730-50-9884|  Giza|   Sports and travel| 610.491|   4.0|
|182-69-8360| Cairo|Electronic access...|   99.33|   4.0|
|730-61-8757| Cairo|   Health and beauty| 214.746|   4.0|
|828-46-6863|  Alex|  Food and beverages| 620.739|   4.0|
|131-15-8856|  Giza|  Food and beverages| 609.168|   4.0|
|576-31-4774| Cairo|   Health and beauty|231.2415|   4.0|
|651-96-5970|  Alex| Fashion accessories| 48.7305|   4.0|
|845-94-6841|  Giza|  Food and beverages| 688.716|   4.0|
+-----------+------+--------------------+--------+------+
only showing top 10 rows



# 8.1 — Creating Derived Columns
`.withColumn()` adds or replaces a column using an expression.

In [ ]:
from pyspark.sql.functions import when, month, dayofweek, lit

df_enriched = df \
    .withColumn("Revenue_Band",
        when(col("Sales") >= 300, "High")
        .when(col("Sales") >= 150, "Medium")
        .otherwise("Low")
    ) \
    .withColumn("Month",      month(col("Date"))) \
    .withColumn("Day_of_Week", dayofweek(col("Date"))) \
    .withColumn("Revenue_per_Item", spark_round(col("Sales") / col("Quantity"), 2))

df_enriched.select("Invoice_ID","Sales","Revenue_Band","Month","Day_of_Week","Revenue_per_Item").show(5)

+-----------+--------+------------+-----+-----------+----------------+
| Invoice_ID|   Sales|Revenue_Band|Month|Day_of_Week|Revenue_per_Item|
+-----------+--------+------------+-----+-----------+----------------+
|750-67-8428|548.9715|        High|    1|          7|           78.42|
|226-31-3081|   80.22|         Low|    3|          6|           16.04|
|631-41-3108|340.5255|        High|    3|          1|           48.65|
|123-19-1176| 489.048|        High|    1|          1|           61.13|
|373-73-7910|634.3785|        High|    2|          6|           90.63|
+-----------+--------+------------+-----+-----------+----------------+
only showing top 5 rows



In [ ]:
df_enriched.groupBy("Revenue_Band").count().orderBy("Revenue_Band").show()

+------------+-----+
|Revenue_Band|count|
+------------+-----+
|        High|  431|
|         Low|  306|
|      Medium|  263|
+------------+-----+



# 9.1 — Spark SQL
Register the DataFrame as a temporary view and query it with standard SQL.

In [ ]:
df_enriched.createOrReplaceTempView("sales")
print("Temp view 'sales' registered")

Temp view 'sales' registered


In [ ]:
spark.sql("""
    SELECT Branch,
           ROUND(SUM(Sales), 2)  AS Total_Revenue,
           ROUND(AVG(Rating), 2) AS Avg_Rating,
           COUNT(*)              AS Transactions
    FROM   sales
    GROUP  BY Branch
    ORDER  BY Total_Revenue DESC
""").show()

+------+-------------+----------+------------+
|Branch|Total_Revenue|Avg_Rating|Transactions|
+------+-------------+----------+------------+
|  Giza|    110568.71|      7.07|         328|
|  Alex|    106200.37|      7.03|         340|
| Cairo|    106197.67|      6.82|         332|
+------+-------------+----------+------------+



In [ ]:
spark.sql("""
    SELECT Product_line,
           SUM(Quantity)         AS Total_Units,
           ROUND(SUM(Sales), 2)  AS Total_Revenue
    FROM   sales
    GROUP  BY Product_line
    ORDER  BY Total_Revenue DESC
""").show()

+--------------------+-----------+-------------+
|        Product_line|Total_Units|Total_Revenue|
+--------------------+-----------+-------------+
|  Food and beverages|        952|     56144.84|
|   Sports and travel|        920|     55122.83|
|Electronic access...|        971|     54337.53|
| Fashion accessories|        902|      54305.9|
|  Home and lifestyle|        911|     53861.91|
|   Health and beauty|        854|     49193.74|
+--------------------+-----------+-------------+



In [ ]:
spark.sql("""
    SELECT Customer_type,
           Revenue_Band,
           COUNT(*)             AS Count,
           ROUND(SUM(Sales), 2) AS Revenue
    FROM   sales
    GROUP  BY Customer_type, Revenue_Band
    ORDER  BY Customer_type, Revenue_Band
""").show()

+-------------+------------+-----+---------+
|Customer_type|Revenue_Band|Count|  Revenue|
+-------------+------------+-----+---------+
|       Member|        High|  257|144707.09|
|       Member|         Low|  163| 13098.72|
|       Member|      Medium|  145| 31888.95|
|       Normal|        High|  174| 95223.94|
|       Normal|         Low|  143| 12389.25|
|       Normal|      Medium|  118| 25658.79|
+-------------+------------+-----+---------+



# 10.1 — Save Enriched DataFrame
Save as **Parquet** (efficient columnar format) and **CSV** for sharing.

In [ ]:
df_enriched.write.mode("overwrite").parquet("enriched_sales.parquet")
df_enriched.write.mode("overwrite").option("header", True).csv("enriched_sales_csv")
print("Saved: enriched_sales.parquet and enriched_sales_csv/")

Saved: enriched_sales.parquet and enriched_sales_csv/


In [ ]:
print("=" * 50)
print("SESSION SUMMARY")
print("=" * 50)
print(f"Original DataFrame  : {df.count():,} rows x {len(df.columns)} cols")
print(f"Enriched DataFrame  : {df_enriched.count():,} rows x {len(df_enriched.columns)} cols")
print(f"New columns added   : Revenue_Band, Month, Day_of_Week, Revenue_per_Item")
print(f"Spark SQL queries   : 3 executed on temp view 'sales'")
print(f"Output files        : enriched_sales.parquet, enriched_sales_csv/")

SESSION SUMMARY
Original DataFrame  : 1,000 rows x 17 cols
Enriched DataFrame  : 1,000 rows x 21 cols
New columns added   : Revenue_Band, Month, Day_of_Week, Revenue_per_Item
Spark SQL queries   : 3 executed on temp view 'sales'
Output files        : enriched_sales.parquet, enriched_sales_csv/
